# Nadir Verifier — DeBERTa-v3-small training on Colab free T4

Trains the verifier-gated cascade discriminator described in `ip-1-verifier-gated-cascade.md`.
Runs on Colab free tier T4 in ~30-60 minutes for a 10K-triple corpus.

**Pipeline**:
1. Install transformers + datasets + supabase (free)
2. Pull labeled training data from `verifier_training_corpus` table
3. Tokenize as cross-encoder input: `[CLS] prompt [SEP] cheap [SEP] expensive [SEP]`
4. Fine-tune DeBERTa-v3-small with binary cross-entropy
5. Eval on the held-out test split
6. INT8 dynamic quantization for CPU inference
7. Download checkpoint to local machine

**Cost**: $0. Free tier T4 + your existing Supabase project.

**Prerequisite**: at least one row in `verifier_training_corpus` with `label IS NOT NULL`. Bootstrap via the OAuth-judge validation cycle first.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers==4.46.0 datasets==3.0.0 accelerate==1.0.0 supabase==2.30.0 scikit-learn

In [ ]:
# Cell 2 — Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "no GPU")

In [ ]:
# Cell 3 — Supabase credentials
# Paste your Supabase URL + service key (use a NOTEBOOK secret, not committed plaintext)
from google.colab import userdata
import os
os.environ['SUPABASE_URL'] = userdata.get('SUPABASE_URL')
os.environ['SUPABASE_SERVICE_KEY'] = userdata.get('SUPABASE_SERVICE_KEY')
print('Supabase creds loaded from Colab secrets.')

In [ ]:
# Cell 4 — Pull labeled triples from Supabase
from supabase import create_client
import os

sb = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SERVICE_KEY'])

# Fetch all labeled triples. Page if more than 1000.
rows = []
page_size = 1000
offset = 0
while True:
    r = sb.table('verifier_training_corpus').select(
        'id, prompt, cheap_answer, expensive_answer, label, split'
    ).not_.is_('label', 'null').range(offset, offset + page_size - 1).execute()
    if not r.data:
        break
    rows.extend(r.data)
    if len(r.data) < page_size:
        break
    offset += page_size

print(f'Loaded {len(rows)} labeled triples.')
print(f"Split distribution: {dict((s, sum(1 for r in rows if r['split']==s)) for s in ('train','val','test'))}")
print(f"Label distribution: {dict((l, sum(1 for r in rows if r['label']==l)) for l in (0, 1))}")

In [ ]:
# Cell 5 — Tokenize as cross-encoder pairs
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'microsoft/deberta-v3-small'
MAX_LEN = 512

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_input(prompt, cheap, expensive):
    # Cross-encoder: [CLS] prompt [SEP] cheap [SEP] expensive [SEP]
    return tok(
        text=prompt,
        text_pair=f"CHEAP:\n{cheap}\n\nEXPENSIVE:\n{expensive}",
        truncation=True,
        max_length=MAX_LEN,
        padding=False,
    )

def to_dataset(filtered_rows):
    examples = []
    for r in filtered_rows:
        enc = make_input(r['prompt'], r['cheap_answer'], r['expensive_answer'])
        enc['labels'] = int(r['label'])
        examples.append(enc)
    return Dataset.from_list(examples)

train_rows = [r for r in rows if r['split'] == 'train']
val_rows   = [r for r in rows if r['split'] == 'val']
test_rows  = [r for r in rows if r['split'] == 'test']

# If no explicit splits, do an 80/10/10 random split
if not train_rows and not val_rows and not test_rows:
    import random
    random.seed(42)
    shuffled = list(rows)
    random.shuffle(shuffled)
    n = len(shuffled)
    train_rows = shuffled[:int(0.8*n)]
    val_rows   = shuffled[int(0.8*n):int(0.9*n)]
    test_rows  = shuffled[int(0.9*n):]
    print(f'Auto-split applied: train={len(train_rows)} val={len(val_rows)} test={len(test_rows)}')

train_ds = to_dataset(train_rows)
val_ds   = to_dataset(val_rows)
test_ds  = to_dataset(test_rows)
print(f'Train: {len(train_ds)} Val: {len(val_ds)} Test: {len(test_ds)}')

In [ ]:
# Cell 6 — Train DeBERTa-v3-small with binary classification head
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import numpy as np
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

data_collator = DataCollatorWithPadding(tokenizer=tok)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float('nan')
    return {'auroc': auc, 'precision': p, 'recall': r, 'f1': f1}

args = TrainingArguments(
    output_dir='./checkpoints',
    num_train_epochs=4,
    per_device_train_batch_size=8,    # T4 has 16GB VRAM; 8 is safe at 512 tokens
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='auroc',
    greater_is_better=True,
    fp16=True,                          # T4 supports fp16; halves memory
    logging_steps=10,
    report_to=[],                        # no wandb / tensorboard
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Cell 7 — Evaluate on test set
test_metrics = trainer.evaluate(eval_dataset=test_ds)
print('Test metrics:')
for k, v in test_metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

# Gate: production deployment requires AUROC >= 0.82 per IP-1 blueprint
if test_metrics.get('eval_auroc', 0) < 0.82:
    print('\nWARNING: AUROC below the 0.82 production gate. Do not deploy this checkpoint as-is.')
    print('Options: more training data, longer training, larger model, or revise judge prompt.')
else:
    print('\nAUROC clears the 0.82 production gate.')

In [ ]:
# Cell 8 — INT8 dynamic quantization for CPU inference
import torch
from pathlib import Path

trainer.save_model('./checkpoints/best')
tok.save_pretrained('./checkpoints/best')

# Load on CPU and quantize
model_cpu = AutoModelForSequenceClassification.from_pretrained('./checkpoints/best').to('cpu').eval()
quantized = torch.quantization.quantize_dynamic(
    model_cpu, {torch.nn.Linear}, dtype=torch.qint8
)
torch.save(quantized.state_dict(), './checkpoints/best/verifier_int8.pt')
print('Quantized weights saved to ./checkpoints/best/verifier_int8.pt')

# Latency benchmark
import time
sample = tok('test prompt', 'CHEAP: foo\n\nEXPENSIVE: bar', return_tensors='pt', truncation=True, max_length=512)
for _ in range(3): quantized(**sample)  # warmup
t0 = time.perf_counter()
N = 50
for _ in range(N): quantized(**sample)
elapsed = (time.perf_counter() - t0) / N * 1000
print(f'CPU inference latency p50 estimate: {elapsed:.1f}ms (target: <50ms)')

In [ ]:
# Cell 9 — Package + download checkpoint
import shutil
shutil.make_archive('/content/verifier_checkpoint', 'zip', './checkpoints/best')
print('Wrote /content/verifier_checkpoint.zip')

# Download to local machine
from google.colab import files
files.download('/content/verifier_checkpoint.zip')
print('\nUnzip locally into getnadir.dev/verifier/weights/ for backend startup loading.')